In [ ]:
import sys
from types import ModuleType

# 제거된 vertexai 모듈을 메모리에 가짜로 등록
dummy_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertex.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertex

dummy_llm_vertex = ModuleType("langchain_community.llms.vertexai")
dummy_llm_vertex.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = dummy_llm_vertex

In [1]:
import pandas as pd
from ragas import EvaluationDataset

eval_raw_data = {
    "user_input": [
        "도서관 주말 운영 시간과 대출 가능 권수는 어떻게 되나요?",
        "시험 기간 평일 도서관 마감 시간은 몇 시인가요?"
    ],
    "reference": [
        "주말은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
        "시험 기간 평일 마감 시간은 22시까지입니다."
    ],
    "retrieved_contexts": [
        [
            "중앙도서관 운영 안내: 주말(토, 일)은 오전 10시부터 오후 5시까지 운영하며, 1인당 최대 5권까지 대출 가능합니다.",
            "중앙도서관 매점 운영 안내: 평일 오전 9시부터 오후 6시까지 운영합니다."
        ],
        [
            "중앙도서관 특별 규정: 시험 기간 평일은 09시부터 22시까지 연장 운영합니다.",
            "도서관 분실물 보관소는 1층 로비 안내실에서 상시 운영됩니다."
        ]
    ]
}

df_eval = pd.DataFrame(eval_raw_data)

ragas_dataset = EvaluationDataset.from_pandas(df_eval)

ragas_dataset

c:\Users\bsis0\Documents\ax_study\12_rag\chapter06_07\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference'], len=2)

In [10]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from ragas.metrics import ContextPrecision, ContextRecall
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

load_dotenv()

# luna 여기서 쓰면 안됨 - temparature 변경 가능한 모델 사용을 위해.

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="none",
    temperature= 0, # 엄격하게!
)

evaluator_llm = LangchainLLMWrapper(model, bypass_temperature=True)

metrics = [
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm = evaluator_llm),
]
evaluation_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    raise_exceptions=True
)

evaluation_result

C:\Users\bsis0\AppData\Local\Temp\ipykernel_47992\2566084478.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\bsis0\AppData\Local\Temp\ipykernel_47992\2566084478.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\bsis0\AppData\Local\Temp\ipykernel_47992\2566084478.py:19: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key=

{'context_precision': 1.0000, 'context_recall': 1.0000}

In [6]:
df_result = evaluation_result.to_pandas()

output_csv_path = "search_eval_report.csv"
df_result.to_csv(output_csv_path, index=False, encoding="utf-8-sig") # csv 한글 encodding

NameError: name 'evaluation_result' is not defined